# 🎬 Module 6: Item-Based Collaborative Filtering
**Movie Recommendation System Using Machine Learning**

---

### 🎯 Objectives
1. Understand the intuition and mathematical formula for **Item-to-Item Cosine Similarity**.
2. Compute cosine similarity efficiently using transposed SciPy sparse matrices.
3. Find movies most similar to any selected movie.
4. Predict user ratings for unseen movies:
   $$\hat{r}_{u,i} = \frac{\sum_{j \in R(u)} s_{i,j} \cdot r_{u,j}}{\sum_{j \in R(u)} |s_{i,j}|}$$
5. Exclude already-watched movies and rank the top-K recommendations.


In [1]:
import os
import sys
import pandas as pd
import numpy as np

sys.path.insert(0, "..")
from src.data_loader import load_dataset
from src.preprocessing import clean_movies, clean_ratings, filter_insufficient_ratings, train_test_split_ratings, build_user_item_matrix
from src.similarity import compute_item_similarity, get_top_similar_items
from src.recommender import MovieRecommender

print("Imports ready!")


Imports ready!


## 1. Mathematical Concept: Item-Item Cosine Similarity

Given two items $i$ and $j$, represented as rating vectors across all users $\mathbf{r}_i, \mathbf{r}_j$:
$$\text{similarity}(i, j) = \cos(\mathbf{r}_i, \mathbf{r}_j) = \frac{\sum_u r_{u,i} \cdot r_{u,j}}{\sqrt{\sum_u r_{u,i}^2} \cdot \sqrt{\sum_u r_{u,j}^2}}$$
- $1.0$ indicates identical co-rating profiles.
- $0.0$ indicates completely orthogonal (no shared users or zero overlap).


In [2]:
# Load pre-trained model or train pipeline
model = MovieRecommender.load("../models/recommender_model.joblib")
print(f"Loaded model with {len(model.movie_to_idx):,} movies and {len(model.user_to_idx):,} users.")


Loaded model with 2,269 movies and 610 users.


## 2. Finding Movies Similar to a Target Movie

In [3]:
def show_similar(movie_title, top_k=5):
    matches = model.search_movies(movie_title, top_k=1)
    if not matches:
        print("Movie not found!")
        return
    m_id = matches[0]["movieId"]
    print(f"🎯 Target Movie: {matches[0]['title']} ({matches[0]['genres']})")
    sims = model.recommend_similar_movies(m_id, top_k=top_k)
    return pd.DataFrame(sims)

display(show_similar("Toy Story", top_k=5))


🎯 Target Movie: Toy Story (1995) (Adventure|Animation|Children|Comedy|Fantasy)


,movieId,title,genres,similarity_score
0,3114,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy,0.5037
1,480,Jurassic Park (1993),Action|Adventure|Sci-Fi|Thriller,0.5019
2,780,Independence Day (a.k.a. ID4) (1996),Action|Adventure|Sci-Fi|Thriller,0.4725
3,364,"Lion King, The (1994)",Adventure|Animation|Children|Drama|Musical|IMAX,0.4684
4,1270,Back to the Future (1985),Adventure|Comedy|Sci-Fi,0.4677


In [4]:
display(show_similar("Matrix, The", top_k=5))


🎯 Target Movie: Matrix, The (1999) (Action|Sci-Fi|Thriller)


,movieId,title,genres,similarity_score
0,2959,Fight Club (1999),Action|Crime|Drama|Thriller,0.5815
1,1210,Star Wars: Episode VI - Return of the Jedi (1983),Action|Adventure|Sci-Fi,0.5605
2,2028,Saving Private Ryan (1998),Action|Drama|War,0.5566
3,1196,Star Wars: Episode V - The Empire Strikes Back...,Action|Adventure|Sci-Fi,0.5534
4,5952,"Lord of the Rings: The Two Towers, The (2002)",Adventure|Fantasy,0.5365


## 3. Generating Personalized Recommendations for a User
We predict ratings for unrated movies and strictly exclude already-watched films.


In [5]:
user_id = 1
user_watched = model.user_watched.get(user_id, set())
print(f"User {user_id} has watched {len(user_watched)} movies in training set.")

recs = model.recommend_item_based(user_id=user_id, top_k=5)
rec_df = pd.DataFrame(recs)
display(rec_df)

# Check exclusion of watched movies
overlaps = set(rec_df["movieId"]).intersection(user_watched)
print(f"Watched movies in recommendation list: {len(overlaps)} (Must be 0)")


User 1 has watched 163 movies in training set.


,movieId,title,genres,predicted_score,method
0,177765,Coco (2017),Adventure|Animation|Children,4.63,item_based_cf
1,150548,Sherlock: The Abominable Bride (2016),Action|Crime|Drama|Mystery|Thriller,4.56,item_based_cf
2,5380,"Importance of Being Earnest, The (2002)",Comedy|Drama|Romance,4.54,item_based_cf
3,5135,Monsoon Wedding (2001),Comedy|Romance,4.53,item_based_cf
4,92259,Intouchables (2011),Comedy|Drama,4.53,item_based_cf


Watched movies in recommendation list: 0 (Must be 0)


## 4. Beginner Exercises
1. Pick your favorite movie from the catalog. What are its top 5 most similar movies according to the cosine similarity matrix?
2. Why is Item-Based Collaborative Filtering generally more stable over time than User-Based CF in large streaming systems like Netflix?
   *(Hint: Item profiles change slowly as new ratings come in; user tastes change dynamically.)*
